In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import logging
import os
import warnings
import itertools
from datetime import datetime, timedelta
from functools import partial
from pathlib import Path
from optimization_engines_v2 import dwave_bqm_qubo, dwave_cqm_qubo, genetic_algorithm_qubo

# Load D-Wave API token from .env file
from dotenv import load_dotenv
load_dotenv()

# Set D-Wave token as environment variable
if 'DWAVE_API_TOKEN' in os.environ:
    os.environ['DWAVE_API_TOKEN'] = os.environ['DWAVE_API_TOKEN']
    print("✅ D-Wave API token loaded successfully")
else:
    print("❌ D-Wave API token not found in .env file")

SEED = 14
np.random.seed(SEED)
msg_level = logging.INFO
# Suppress all RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

## D-Wave Connection Test: QUBO Optimization (10 Stocks, Single Period)

In [ ]:
# Create a logger
logger = logging.getLogger("dwave_test_logger")
logger.setLevel(msg_level)  # Set the level for this logger

# Create a handler (where to send the logs)
handler = logging.StreamHandler()  # Send to the console
handler.setLevel(msg_level)

# Create a formatter (how to format the logs)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(handler)

### Path definition

In [ ]:
benchmark_path = '../../data/benchmark_gspc.pkl'
source_path = '../../data/stocks_adjclose.pkl'

### Data loading

In [ ]:
benchmark = pd.read_pickle(benchmark_path)
sns.lineplot(benchmark['^GSPC'])
benchmark.head()

In [ ]:
source = pd.read_pickle(source_path)
print(source.shape)
# Check if any value in the DataFrame is null
has_any_nan = source.isnull().values.any()
print("Any NaN in source:", has_any_nan)
source.head()

### Select 10 stocks for connection test

In [ ]:
# Simplified stock selection - rank by correlation and returns
df_corr = source.corr()
corr_sum = df_corr.map(lambda x: abs(x)).sum()
corr_rank = corr_sum.sort_values().rank(method='min').astype(int)

return_rank = source.diff().sum(axis=0).sort_values().rank(method='min', ascending=False).astype(int)
select_10 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:10]
print(f"Selected 10 stocks for test: {select_10}")

### Portfolio Stats

In [ ]:
def portfolio_stats(weights, data):
    weights = np.array(weights)
    returns = np.log(data) - np.log(data.shift(1)) # log return to minimize fp error
    port_return = np.sum(returns.mean() * weights) 
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() , weights)))
    try:
        sharpe_ratio = port_return/port_vol
    except Exception as e:
        sharpe_ratio = 0
    return sharpe_ratio, port_return, port_vol

def qubo_fitness_function(weights, data, risk_level=0.5):
    """
    QUBO fitness function with fixed risk level.
    Maximizes: return - risk_level * variance
    """
    port_return, port_vol = portfolio_stats(weights, data)[:1] + portfolio_stats(weights, data)[1:]
    portfolio_variance = port_vol ** 2
    objective = port_return - risk_level * portfolio_variance
    return objective

### Simple Single-Period Test Function

In [ ]:
def single_period_test(optimization_function, data, test_name):
    """
    Simple single-period optimization test.
    
    Args:
        optimization_function: Function to test
        data: Stock price data
        test_name: Name for logging
    
    Returns:
        results: Dictionary with weights, stats, and timing
    """
    print(f"\n{'='*60}")
    print(f"TESTING: {test_name}")
    print(f"{'='*60}")
    
    # Use most recent 60 days of data for test
    test_data = data.iloc[-60:]
    
    try:
        start_time = datetime.now()
        print(f"Starting optimization at {start_time}...")
        
        # Run optimization
        weights = optimization_function(test_data)
        
        end_time = datetime.now()
        execution_time = (end_time - start_time).total_seconds()
        
        # Calculate portfolio statistics
        sharpe, port_return, port_vol = portfolio_stats(weights, test_data)
        qubo_objective = qubo_fitness_function(weights, test_data, risk_level=0.5)
        
        # Results summary
        results = {
            'test_name': test_name,
            'weights': weights,
            'sharpe_ratio': sharpe,
            'portfolio_return': port_return,
            'portfolio_volatility': port_vol,
            'qubo_objective': qubo_objective,
            'execution_time': execution_time,
            'success': True,
            'error': None
        }
        
        print(f"✅ SUCCESS: {test_name}")
        print(f"   Execution time: {execution_time:.2f}s")
        print(f"   Sharpe ratio: {sharpe:.4f}")
        print(f"   QUBO objective: {qubo_objective:.4f}")
        print(f"   Portfolio return: {port_return:.4f}")
        print(f"   Portfolio volatility: {port_vol:.4f}")
        print(f"   Weights sum: {weights.sum():.6f}")
        print(f"   All weights > 0: {all(w > 0 for w in weights)}")
        
        return results
        
    except Exception as e:
        end_time = datetime.now()
        execution_time = (end_time - start_time).total_seconds()
        
        results = {
            'test_name': test_name,
            'weights': None,
            'sharpe_ratio': None,
            'portfolio_return': None,
            'portfolio_volatility': None,
            'qubo_objective': None,
            'execution_time': execution_time,
            'success': False,
            'error': str(e)
        }
        
        print(f"❌ FAILED: {test_name}")
        print(f"   Execution time: {execution_time:.2f}s")
        print(f"   Error: {e}")
        
        return results

### Test Parameters

In [ ]:
# Parameters for single-period connection test
test_parameters = {
    # BQM/CQM parameters
    "budget": 1.0,
    "min_investment": 0.001,
    
    # GA parameters  
    "population_size": 100,
    "num_generations": 50,
    "mutation_rate": 0.1,
    "elitism": 0.1,
}

print(f"Test parameters: {test_parameters}")

### Connection Test: 10 Stocks Dataset

In [ ]:
# Prepare 10 stocks dataset
data_10 = source[select_10]
print(f"Test dataset shape: {data_10.shape}")
print(f"Date range: {data_10.index[0]} to {data_10.index[-1]}")

# Plot the data
plt.figure(figsize=(12,6))
sns.set_style('darkgrid')
data_10.plot(figsize=(12,6))
plt.title('10 Stocks Test Dataset')
plt.legend(loc='upper left')
plt.show()

#### Test 1: D-Wave BQM QUBO Optimization

In [ ]:
# Setup BQM optimization function
bqm_optimizer = partial(
    dwave_bqm_qubo, 
    budget=test_parameters["budget"], 
    min_investment=test_parameters["min_investment"]
)

# Run BQM test
bqm_results = single_period_test(
    bqm_optimizer, 
    data_10, 
    "D-Wave BQM QUBO Optimization"
)

#### Test 2: D-Wave CQM QUBO Optimization

In [ ]:
# Setup CQM optimization function
cqm_optimizer = partial(
    dwave_cqm_qubo, 
    budget=test_parameters["budget"], 
    min_investment=test_parameters["min_investment"]
)

# Run CQM test
cqm_results = single_period_test(
    cqm_optimizer, 
    data_10, 
    "D-Wave CQM QUBO Optimization"
)

#### Test 3: Genetic Algorithm QUBO Optimization (Baseline)

In [ ]:
# Setup GA optimization function
ga_optimizer = partial(
    genetic_algorithm_qubo,
    population_size=test_parameters["population_size"],
    num_generations=test_parameters["num_generations"],
    mutation_rate=test_parameters["mutation_rate"],
    elitism=test_parameters["elitism"]
)

# Run GA test
ga_results = single_period_test(
    ga_optimizer,
    data_10,
    "Genetic Algorithm QUBO Optimization"
)

#### Comparison Results

In [ ]:
# Compare results
print("\n" + "="*80)
print("CONNECTION TEST RESULTS SUMMARY")
print("="*80)

results = [bqm_results, cqm_results, ga_results]
comparison_df = pd.DataFrame([
    {
        'Method': r['test_name'],
        'Success': '✅ Yes' if r['success'] else '❌ No',
        'Time (s)': f"{r['execution_time']:.2f}" if r['execution_time'] else 'N/A',
        'QUBO Obj': f"{r['qubo_objective']:.4f}" if r['qubo_objective'] is not None else 'N/A',
        'Sharpe': f"{r['sharpe_ratio']:.4f}" if r['sharpe_ratio'] is not None else 'N/A',
        'Return': f"{r['portfolio_return']:.4f}" if r['portfolio_return'] is not None else 'N/A',
        'Volatility': f"{r['portfolio_volatility']:.4f}" if r['portfolio_volatility'] is not None else 'N/A',
        'Error': r['error'] if r['error'] else 'None'
    }
    for r in results
])

display(comparison_df)

# Connection status
print("\nD-WAVE CONNECTION STATUS:")
print("-" * 40)
if bqm_results['success']:
    print("🟢 D-Wave BQM connection: SUCCESSFUL")
    print(f"   ⏱️  Execution time: {bqm_results['execution_time']:.2f}s")
    print(f"   📊 QUBO objective achieved: {bqm_results['qubo_objective']:.4f}")
else:
    print("🔴 D-Wave BQM connection: FAILED")
    print(f"   ❌ Error: {bqm_results['error']}")

if cqm_results['success']:
    print("🟢 D-Wave CQM connection: SUCCESSFUL")
    print(f"   ⏱️  Execution time: {cqm_results['execution_time']:.2f}s")
    print(f"   📊 QUBO objective achieved: {cqm_results['qubo_objective']:.4f}")
else:
    print("🔴 D-Wave CQM connection: FAILED")
    print(f"   ❌ Error: {cqm_results['error']}")

# Performance comparison (if all successful)
successful_results = [r for r in results if r['success']]
if len(successful_results) >= 2:
    print("\nPERFORMANCE COMPARISON:")
    print("-" * 40)
    
    # Find best QUBO objective
    best_qubo = max(successful_results, key=lambda x: x['qubo_objective'])
    print(f"🏆 Best QUBO objective: {best_qubo['test_name']} ({best_qubo['qubo_objective']:.4f})")
    
    # Find fastest
    fastest = min(successful_results, key=lambda x: x['execution_time'])
    print(f"🚀 Fastest execution: {fastest['test_name']} ({fastest['execution_time']:.2f}s)")
    
    # Find best Sharpe
    best_sharpe = max(successful_results, key=lambda x: x['sharpe_ratio'])
    print(f"📈 Best Sharpe ratio: {best_sharpe['test_name']} ({best_sharpe['sharpe_ratio']:.4f})")
        
print("\n" + "="*80)
print("TEST COMPLETED")
print("="*80)

#### Weights Comparison (if methods successful)

In [ ]:
# Compare portfolio weights if methods are successful
successful_results = [r for r in results if r['success']]
if len(successful_results) >= 2:
    # Create weights comparison DataFrame
    weights_data = {'Stock': select_10}
    
    for result in successful_results:
        method_name = result['test_name'].split()[1] + '_' + result['test_name'].split()[2]  # Extract method name
        weights_data[f'{method_name}_Weights'] = result['weights']
    
    weights_df = pd.DataFrame(weights_data)
    weights_df = weights_df.round(4)
    
    print("PORTFOLIO WEIGHTS COMPARISON:")
    display(weights_df)
    
    # Visualize weights comparison
    n_methods = len(successful_results)
    fig, axes = plt.subplots(1, min(2, n_methods), figsize=(15, 6))
    if n_methods == 1:
        axes = [axes]
    
    # Bar plot comparison
    if n_methods >= 2:
        x = np.arange(len(select_10))
        width = 0.8 / n_methods
        
        for i, result in enumerate(successful_results):
            method_name = result['test_name'].split()[:2]  # First two words
            method_label = ' '.join(method_name)
            axes[0].bar(x + i*width - width*(n_methods-1)/2, result['weights'], 
                       width, label=method_label, alpha=0.8)
        
        axes[0].set_xlabel('Stocks')
        axes[0].set_ylabel('Portfolio Weights')
        axes[0].set_title('Portfolio Weights Comparison')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(select_10, rotation=45)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # QUBO objective vs weights scatter
        if len(axes) > 1:
            for result in successful_results:
                method_name = result['test_name'].split()[:2]
                method_label = ' '.join(method_name)
                axes[1].scatter([result['qubo_objective']] * len(result['weights']), 
                               result['weights'], label=method_label, alpha=0.7, s=60)
            
            axes[1].set_xlabel('QUBO Objective')
            axes[1].set_ylabel('Individual Stock Weights')
            axes[1].set_title('QUBO Objective vs Stock Weights')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlations between weight allocations if 2+ methods
    if len(successful_results) >= 2:
        print("\nWeights correlations between methods:")
        for i in range(len(successful_results)):
            for j in range(i+1, len(successful_results)):
                method1 = successful_results[i]['test_name'].split()[:2]
                method2 = successful_results[j]['test_name'].split()[:2]
                correlation = np.corrcoef(successful_results[i]['weights'], 
                                        successful_results[j]['weights'])[0, 1]
                print(f"  {' '.join(method1)} vs {' '.join(method2)}: {correlation:.4f}")
    
else:
    print("Cannot compare weights - insufficient successful optimization methods.")